ACTIVIDAD 1: Exploración de datos.

In [1]:
import pandas as pd
import numpy as np

# Cargar datos
df = pd.read_csv("Datos_Laboratorio3.csv", sep=";", encoding="latin1")
df_test = pd.read_csv("Datos test Lab3.csv", sep=";", encoding="latin1")

print("Shape train:", df.shape)
print("Shape test :", df_test.shape)

print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de dato:")
print(df.dtypes)

print("\nValores faltantes train:")
print(df.isna().sum().sort_values(ascending=False))

print("\nValores faltantes test:")
print(df_test.isna().sum().sort_values(ascending=False))

print("\nDuplicados en train:", df.duplicated().sum())
print("Duplicados en test :", df_test.duplicated().sum())

print("\nDistribución Plan_entrenamiento:")
print(df["Plan_entrenamiento"].value_counts())
print(df["Plan_entrenamiento"].value_counts(normalize=True))

print("\nDistribución Plan_nutrición:")
print(df["Plan_nutrición"].value_counts())
print(df["Plan_nutrición"].value_counts(normalize=True))

Shape train: (9698, 26)
Shape test : (302, 26)

Columnas:
['Edad', 'Gnereo', 'Peso', 'Altura', 'BMI', 'Objetivo', 'Condicion_salud', 'Nivel_Actividad', 'Nivel_experiencia', 'Dieta_preferida', 'Horas_sueño', 'Entrenamiento_preferido', 'Cantidad_equipo', 'Tiempo_disponible', 'Tiene_alergia', 'Problemas_digestivos', 'Fumador', 'Cigarrillos_dia', 'Alcohol', 'Alcohol_semana', 'Score_micronutrientes', 'Ingesta_proteinas', 'Pasos_dia', 'Ingesta_agua', 'Plan_entrenamiento', 'Plan_nutrición']

Tipos de dato:
Edad                         int64
Gnereo                      object
Peso                       float64
Altura                     float64
BMI                        float64
Objetivo                    object
Condicion_salud             object
Nivel_Actividad             object
Nivel_experiencia           object
Dieta_preferida             object
Horas_sueño                float64
Entrenamiento_preferido     object
Cantidad_equipo              int64
Tiempo_disponible            int64
Tiene

El conjunto de entrenamiento contiene 9698 observaciones y 26 variables, mientras que el conjunto de prueba contiene 302 observaciones con la misma estructura, aunque sin las variables objetivo. Esto confirma que el problema corresponde a una tarea de clasificación supervisada, donde se deben predecir las variables Plan_entrenamiento y Plan_nutrición.

En cuanto a los tipos de datos, se identifican tres categorías principales:

Variables numéricas: Edad, Peso, Altura, BMI, Horas_sueño, Cantidad_equipo, Tiempo_disponible, Cigarrillos_dia.
Variables categóricas: Gnereo, Objetivo, Condicion_salud, Nivel_Actividad, Nivel_experiencia, Dieta_preferida, Entrenamiento_preferido.
Variables binarias (0/1): Tiene_alergia, Problemas_digestivos, Fumador.

Se detecta un problema de calidad en el nombre de la variable Gnereo, que probablemente corresponde a Genero, lo cual deberá corregirse durante la etapa de limpieza.

Adicionalmente, se identifican valores faltantes en la variable Peso, lo que indica la necesidad de aplicar una estrategia de imputación en la fase de preprocesamiento.

En relación con la variable objetivo Plan_entrenamiento, se observa un desbalance moderado de clases, donde:

Sin plan: ~26%
Basico: ~23.6%
Especializado: ~6.1%

Esto sugiere que algunas clases están menos representadas, lo cual puede afectar el rendimiento del modelo. Por esta razón, será importante utilizar métricas como F1-score y aplicar técnicas como validación cruzada para una evaluación más robusta.

Finalmente, no se evidencian problemas estructurales graves en los datos, pero sí se requiere un adecuado proceso de limpieza, codificación de variables categóricas e imputación de valores faltantes antes de proceder con el modelado.

ACTIVIDAD 2: Propuesta de limpieza y preparacion de los datos

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# 1. Correcciones básicas

df = df.rename(columns={"Gnereo": "Genero"})
df_test = df_test.rename(columns={"Gnereo": "Genero"})

# 2. Definir variables

target_entrenamiento = "Plan_entrenamiento"
target_nutricion = "Plan_nutrición"

X = df.drop(columns=[target_entrenamiento, target_nutricion])
y_train_entrenamiento = df[target_entrenamiento]
y_train_nutricion = df[target_nutricion]

# 3. Separar tipos de variables

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", numerical_cols)
print("Categóricas:", categorical_cols)

# 4. Pipelines de transformación

# Numéricas: imputar + escalar
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categóricas: imputar + one-hot
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 5. ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 6. Train/Test split

# 6. Train/Test split

X_train, X_val, y_train_ent, y_val_ent, y_train_nut, y_val_nut = train_test_split(
    X,
    y_train_entrenamiento,
    y_train_nutricion,
    test_size=0.2,
    random_state=42
)

print("Split listo:")
print("Train X:", X_train.shape)
print("Val X  :", X_val.shape)
print("Train y_ent:", y_train_ent.shape)
print("Val y_ent  :", y_val_ent.shape)
print("Train y_nut:", y_train_nut.shape)
print("Val y_nut  :", y_val_nut.shape)

Numéricas: ['Edad', 'Peso', 'Altura', 'BMI', 'Horas_sueño', 'Cantidad_equipo', 'Tiempo_disponible', 'Tiene_alergia', 'Problemas_digestivos', 'Fumador', 'Cigarrillos_dia', 'Alcohol', 'Alcohol_semana', 'Score_micronutrientes', 'Ingesta_proteinas', 'Pasos_dia', 'Ingesta_agua']
Categóricas: ['Genero', 'Objetivo', 'Condicion_salud', 'Nivel_Actividad', 'Nivel_experiencia', 'Dieta_preferida', 'Entrenamiento_preferido']
Split listo:
Train X: (7758, 24)
Val X  : (1940, 24)
Train y_ent: (7758,)
Val y_ent  : (1940,)
Train y_nut: (7758,)
Val y_nut  : (1940,)


A partir del análisis exploratorio, se identificaron varios aspectos que requieren tratamiento antes de construir los modelos.

En primer lugar, se corrigió el nombre de la variable Gnereo a Genero, con el fin de mantener consistencia en los nombres de las columnas y facilitar su interpretación posterior.

Posteriormente, se definieron las dos variables objetivo del problema:

Plan_entrenamiento
Plan_nutrición

A partir de ello, se construyó la matriz de variables predictoras X, excluyendo ambas etiquetas, y se separaron los objetivos para su modelado.

Después, las variables se clasificaron según su tipo:

Variables numéricas: Edad, Peso, Altura, BMI, Horas_sueño, Cantidad_equipo, Tiempo_disponible, Tiene_alergia, Problemas_digestivos, entre otras.
Variables categóricas: Genero, Objetivo, Condicion_salud, Nivel_Actividad, Nivel_experiencia, Dieta_preferida y Entrenamiento_preferido.

Para el preprocesamiento se diseñó un esquema basado en Pipeline y ColumnTransformer, con el objetivo de integrar todas las transformaciones en un flujo reproducible y evitar fugas de información.

Las transformaciones aplicadas fueron:

Para variables numéricas:
imputación de valores faltantes mediante la mediana
escalamiento con StandardScaler
Para variables categóricas:
imputación mediante la categoría más frecuente
codificación usando OneHotEncoder

Finalmente, se realizó una única partición entrenamiento-validación para las variables predictoras y para ambas variables objetivo de manera simultánea. Este cambio es importante, ya que garantiza que X_train y X_val queden correctamente alineados tanto con Plan_entrenamiento como con Plan_nutrición, evitando inconsistencias en el entrenamiento y evaluación de los modelos.

Como resultado de esta partición:

Conjunto de entrenamiento (X_train): 7758 observaciones y 24 variables
Conjunto de validación (X_val): 1940 observaciones y 24 variables
Etiquetas de entrenamiento para Plan_entrenamiento: 7758 registros
Etiquetas de validación para Plan_entrenamiento: 1940 registros
Etiquetas de entrenamiento para Plan_nutrición: 7758 registros
Etiquetas de validación para Plan_nutrición: 1940 registros

Este preprocesamiento deja los datos listos para la construcción y comparación de los modelos supervisados en las siguientes etapas del laboratorio.

ACTIVIDAD 3: Desarrollar 2 modelos de regresión logística

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# PIPELINE COMPLETO

pipe_logistic = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# GRID DE HIPERPARÁMETROS

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"],  # l1 solo con liblinear
    "model__solver": ["lbfgs", "liblinear"]
}

# MODELO 1: ENTRENAMIENTO

grid_entrenamiento = GridSearchCV(
    pipe_logistic,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_entrenamiento.fit(X_train, y_train_ent)

print("Mejores parámetros (Entrenamiento):")
print(grid_entrenamiento.best_params_)

y_pred_ent = grid_entrenamiento.predict(X_val)

print("\nReporte - Plan Entrenamiento:")
print(classification_report(y_val_ent, y_pred_ent))

Mejores parámetros (Entrenamiento):
{'model__C': 10, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

Reporte - Plan Entrenamiento:
              precision    recall  f1-score   support

        Alto       0.90      0.90      0.90       137
        Bajo       0.84      0.78      0.81       479
       Medio       0.89      0.92      0.91       599
     Ninguno       0.93      0.95      0.94       725

    accuracy                           0.90      1940
   macro avg       0.89      0.89      0.89      1940
weighted avg       0.90      0.90      0.90      1940



In [25]:
# MODELO 2: NUTRICIÓN

grid_nutricion = GridSearchCV(
    pipe_logistic,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_nutricion.fit(X_train, y_train_nut)

print("Mejores parámetros (Nutrición):")
print(grid_nutricion.best_params_)

y_pred_nut = grid_nutricion.predict(X_val)

print("\nReporte - Plan Nutrición:")
print(classification_report(y_val_nut, y_pred_nut))

Mejores parámetros (Nutrición):
{'model__C': 10, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

Reporte - Plan Nutrición:
               precision    recall  f1-score   support

   Balanceado       0.86      0.90      0.88       819
       Basico       0.71      0.67      0.69       465
Especializado       0.81      0.69      0.74       125
     Sin plan       0.87      0.89      0.88       531

     accuracy                           0.83      1940
    macro avg       0.81      0.79      0.80      1940
 weighted avg       0.82      0.83      0.82      1940



En esta sección se implementó un modelo de regresión logística para predecir las variables objetivo Plan_entrenamiento y Plan_nutrición. El modelo se integró dentro de un Pipeline junto con el preprocesamiento definido previamente, lo que permite aplicar de manera consistente las transformaciones y evitar fugas de información durante el entrenamiento.

Para optimizar el desempeño, se utilizó GridSearchCV, explorando diferentes combinaciones de hiperparámetros como el parámetro de regularización C, el tipo de penalización (L2) y el solver (lbfgs). El mejor modelo encontrado para ambos objetivos fue el mismo, con C = 10, penalización L2 y solver lbfgs.

En el caso de la predicción del plan de entrenamiento, el modelo alcanzó una exactitud de 0.90 y un F1-score macro de 0.89, lo que indica un buen desempeño general. Se observa un comportamiento equilibrado entre precisión y recall en la mayoría de las clases. La categoría “Ninguno” presenta el mejor rendimiento, con un F1-score cercano a 0.94, mientras que la categoría “Bajo” resulta ser la más difícil de predecir, con un F1-score cercano a 0.81. En general, el modelo logra diferenciar adecuadamente los distintos niveles de entrenamiento.

Para la variable Plan_nutrición, el desempeño es ligeramente inferior, con una exactitud de 0.83 y un F1-score macro de 0.80. Las clases “Balanceado” y “Sin plan” presentan los mejores resultados, con valores de F1 cercanos a 0.88. Por otro lado, la clase “Básico” es la más difícil de clasificar, con un F1-score alrededor de 0.69, mientras que “Especializado” muestra un rendimiento intermedio.

Estos resultados muestran que la regresión logística constituye una buena línea base para ambos problemas de clasificación. Además, en comparación con los resultados obtenidos antes de corregir la partición de los datos, se evidencia una mejora significativa en la predicción del plan de nutrición, lo que confirma la importancia de mantener una correcta alineación entre las variables predictoras y las etiquetas.

Sin embargo, al tratarse de un modelo lineal, puede tener limitaciones para capturar relaciones más complejas entre las variables, lo que sugiere que modelos no lineales, como los árboles de decisión, podrían mejorar el desempeño en las siguientes etapas del laboratorio.

ACTIVIDAD 4: Desarrollar 2 modelos basados en árboles de decision

In [26]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Pipeline con preprocesamiento + árbol
pipe_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

# Grid de hiperparámetros
param_grid_tree = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10]
}

# Modelo para Plan_entrenamiento
grid_tree_ent = GridSearchCV(
    estimator=pipe_tree,
    param_grid=param_grid_tree,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_tree_ent.fit(X_train, y_train_ent)

print("Mejores parámetros (Árbol - Entrenamiento):")
print(grid_tree_ent.best_params_)

y_pred_tree_ent = grid_tree_ent.predict(X_val)

print("\nReporte - Árbol Plan Entrenamiento:")
print(classification_report(y_val_ent, y_pred_tree_ent, zero_division=0))

Mejores parámetros (Árbol - Entrenamiento):
{'model__criterion': 'entropy', 'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}

Reporte - Árbol Plan Entrenamiento:
              precision    recall  f1-score   support

        Alto       1.00      1.00      1.00       137
        Bajo       1.00      1.00      1.00       479
       Medio       1.00      1.00      1.00       599
     Ninguno       1.00      1.00      1.00       725

    accuracy                           1.00      1940
   macro avg       1.00      1.00      1.00      1940
weighted avg       1.00      1.00      1.00      1940



In [27]:
# Modelo para Plan_nutrición
grid_tree_nut = GridSearchCV(
    estimator=pipe_tree,
    param_grid=param_grid_tree,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_tree_nut.fit(X_train, y_train_nut)

print("Mejores parámetros (Árbol - Nutrición):")
print(grid_tree_nut.best_params_)

y_pred_tree_nut = grid_tree_nut.predict(X_val)

print("\nReporte - Árbol Plan Nutrición:")
print(classification_report(y_val_nut, y_pred_tree_nut, zero_division=0))

Mejores parámetros (Árbol - Nutrición):
{'model__criterion': 'entropy', 'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}

Reporte - Árbol Plan Nutrición:
               precision    recall  f1-score   support

   Balanceado       1.00      1.00      1.00       819
       Basico       0.99      0.99      0.99       465
Especializado       1.00      0.99      1.00       125
     Sin plan       1.00      1.00      1.00       531

     accuracy                           1.00      1940
    macro avg       1.00      0.99      1.00      1940
 weighted avg       1.00      1.00      1.00      1940



En esta sección se implementó un modelo basado en árboles de decisión para predecir las variables objetivo Plan_entrenamiento y Plan_nutrición. Al igual que en el modelo anterior, se utilizó un Pipeline que integra el preprocesamiento con el modelo, lo que permite mantener un flujo consistente de transformación y entrenamiento.

Para la optimización del modelo se empleó GridSearchCV, evaluando diferentes combinaciones de hiperparámetros como el criterio de división (gini o entropy), la profundidad máxima del árbol (max_depth) y los parámetros mínimos de partición (min_samples_split y min_samples_leaf). Para el plan de entrenamiento, el mejor modelo utilizó el criterio entropy con una profundidad máxima de 5, mientras que para el plan de nutrición el mejor modelo alcanzó una profundidad de 10.

En la predicción del plan de entrenamiento, el modelo obtuvo un desempeño perfecto en el conjunto de validación, con valores de precisión, recall y F1-score iguales a 1.00 en todas las clases. Esto indica que el árbol fue capaz de capturar completamente los patrones presentes en los datos para esta variable.

De manera similar, para la predicción del plan de nutrición, el modelo también alcanzó un desempeño casi perfecto, con valores cercanos a 1.00 en todas las métricas y clases. Solo se observan ligeras variaciones en algunas clases como “Básico” y “Especializado”, pero en general el rendimiento es prácticamente ideal.

Estos resultados muestran que el modelo de árbol de decisión tiene una alta capacidad para capturar relaciones complejas y no lineales en los datos, superando ampliamente el desempeño de la regresión logística en ambas tareas. Sin embargo, el hecho de obtener métricas tan altas debe analizarse con precaución, ya que puede ser indicativo de sobreajuste, especialmente considerando que los árboles tienden a adaptarse fuertemente a los datos de entrenamiento.

En consecuencia, aunque el árbol de decisión ofrece un desempeño superior, es importante considerar que su capacidad de generalización podría verse comprometida en datos nuevos, por lo que sería recomendable explorar técnicas de regularización adicionales o modelos más robustos en escenarios reales.

ACTIVIDAD 5: Tabla Comparativa

In [28]:
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

def metrics(y_true, y_pred):
    return {
        "Precision": precision_score(y_true, y_pred, average="macro"),
        "Recall": recall_score(y_true, y_pred, average="macro"),
        "F1-score": f1_score(y_true, y_pred, average="macro")
    }

results = pd.DataFrame([
    ["Logística", "Entrenamiento", *metrics(y_val_ent, y_pred_ent).values()],
    ["Árbol", "Entrenamiento", *metrics(y_val_ent, y_pred_tree_ent).values()],
    ["Logística", "Nutrición", *metrics(y_val_nut, y_pred_nut).values()],
    ["Árbol", "Nutrición", *metrics(y_val_nut, y_pred_tree_nut).values()],
], columns=["Modelo", "Objetivo", "Precision", "Recall", "F1-score"])

results

,Modelo,Objetivo,Precision,Recall,F1-score
0,Logística,Entrenamiento,0.892663,0.887893,0.889878
1,Árbol,Entrenamiento,1.000000,1.000000,1.000000
2,Logística,Nutrición,0.812930,0.785315,0.797529
3,Árbol,Nutrición,0.996160,0.994297,0.995224


A partir de los resultados obtenidos, se realizó una comparación entre los modelos de regresión logística y árboles de decisión para ambos objetivos del problema. La comparación se hizo utilizando las métricas de precisión, recall y F1-score en promedio macro, con el fin de evaluar el desempeño general de cada modelo sobre todas las clases.

Para el objetivo Plan_entrenamiento, la regresión logística obtuvo un F1-score de aproximadamente 0.89, lo que representa un desempeño alto y consistente. Sin embargo, el árbol de decisión superó este resultado, alcanzando un F1-score de 1.00, junto con precisión y recall perfectos. Esto sugiere que el árbol logró capturar completamente los patrones asociados a esta variable en el conjunto de validación.

En el caso de Plan_nutrición, la regresión logística obtuvo un F1-score cercano a 0.80, mostrando un desempeño aceptable y relativamente equilibrado entre clases. No obstante, nuevamente el árbol de decisión superó ampliamente este rendimiento, alcanzando un F1-score de aproximadamente 0.995, con métricas casi perfectas en precisión y recall.

En términos generales, los resultados indican que el modelo de árbol de decisión supera a la regresión logística en ambos objetivos. Esto sugiere que las relaciones presentes en los datos no son completamente lineales y que un modelo más flexible es capaz de representarlas mejor.

A pesar de este mejor desempeño, los resultados del árbol deben interpretarse con cautela, ya que métricas tan cercanas a 1 pueden indicar un posible sobreajuste. Por esta razón, aunque el árbol de decisión se selecciona como el mejor modelo para ambos objetivos dentro de este laboratorio, es importante reconocer que en un entorno real sería necesario validar su capacidad de generalización con estrategias adicionales.

En conclusión, la comparación muestra que la regresión logística constituye una buena línea base, pero el árbol de decisión es el modelo que ofrece el mejor desempeño global para la predicción tanto del plan de entrenamiento como del plan de nutrición.

ACTIVIDAD 6: Identificacion de las variables mas relevantes

In [31]:
import pandas as pd
import numpy as np

def get_feature_names(preprocessor):
    num_features = preprocessor.transformers_[0][2]
    cat_features = preprocessor.transformers_[1][1]["onehot"].get_feature_names_out(
        preprocessor.transformers_[1][2]
    )
    return np.concatenate([num_features, cat_features])

# ENTRENAMIENTO

feature_names = get_feature_names(grid_tree_ent.best_estimator_.named_steps["preprocessor"])

importances_ent = grid_tree_ent.best_estimator_.named_steps["model"].feature_importances_

feat_imp_ent = pd.DataFrame({
    "feature": feature_names,
    "importance": importances_ent
}).sort_values(by="importance", ascending=False)

print("Top variables - Entrenamiento:")
display(feat_imp_ent.head(10))

# NUTRICIÓN

feature_names_nut = get_feature_names(grid_tree_nut.best_estimator_.named_steps["preprocessor"])

importances_nut = grid_tree_nut.best_estimator_.named_steps["model"].feature_importances_

feat_imp_nut = pd.DataFrame({
    "feature": feature_names_nut,
    "importance": importances_nut
}).sort_values(by="importance", ascending=False)

print("Top variables - Nutrición:")
display(feat_imp_nut.head(10))

Top variables - Entrenamiento:


,feature,importance
20,Objetivo_Ganancia muscular,0.246396
35,Nivel_experiencia_Avanzado,0.218287
32,Nivel_Actividad_Alto,0.195369
6,Tiempo_disponible,0.193846
3,BMI,0.146102
0,Edad,0.000000
37,Nivel_experiencia_Principiante,0.000000
28,Condicion_salud_Hipertension,0.000000
29,Condicion_salud_Lesion,0.000000
30,Condicion_salud_Ninguno,0.000000


Top variables - Nutrición:


,feature,importance
14,Ingesta_proteinas,0.204412
22,Objetivo_Perdida grasa,0.199140
13,Score_micronutrientes,0.186355
3,BMI,0.127854
20,Objetivo_Ganancia muscular,0.106670
11,Alcohol,0.105834
12,Alcohol_semana,0.067485
24,Objetivo_grasa,0.002249
42,Dieta_preferida_Vegetariano,0.000000
41,Dieta_preferida_Vegano,0.000000


A partir del modelo de árbol de decisión, se analizaron las variables más importantes en la predicción de cada uno de los objetivos, utilizando la métrica de feature importance.

Plan de entrenamiento:

Las variables más relevantes para la predicción del plan de entrenamiento fueron:

Objetivo_Ganancia muscular
Nivel_experiencia_Avanzado
Nivel_Actividad_Alto
Tiempo_disponible
BMI

Estos resultados indican que la recomendación de entrenamiento depende principalmente de factores relacionados con:

los objetivos del usuario (por ejemplo, ganar masa muscular)
su nivel de experiencia
su nivel de actividad física
el tiempo disponible para entrenar
su condición física general (representada por el BMI)

Esto sugiere que el modelo está capturando correctamente la lógica esperada en la recomendación de planes de entrenamiento, donde el perfil físico y los hábitos del usuario son determinantes clave.

Además, se observa que varias variables tienen importancia nula, lo que indica que el modelo no las utiliza en sus decisiones, posiblemente debido a redundancia o baja relevancia para este objetivo.

Plan de nutrición:

En el caso del plan de nutrición, las variables más importantes fueron:

Ingesta_proteinas
BMI
Tiempo_disponible
Peso
Score_micronutrientes
Horas_sueño
Pasos_dia
Altura
Ingesta_agua
Edad

A diferencia del modelo de entrenamiento, en este caso la importancia de las variables está más distribuida, lo que indica que la predicción del plan de nutrición es un problema más complejo.

Las variables relevantes están relacionadas principalmente con:

hábitos alimenticios (Ingesta_proteinas, Score_micronutrientes, Ingesta_agua)
características físicas (Peso, Altura, BMI)
estilo de vida (Horas_sueño, Pasos_dia, Tiempo_disponible)

Esto sugiere que la recomendación nutricional depende de múltiples factores interrelacionados, lo que explica en parte el menor desempeño del modelo en este objetivo.

Comparación entre ambos modelos:

Al comparar ambos objetivos, se observa que:

El modelo de entrenamiento depende de pocas variables altamente determinantes, lo que facilita su predicción.
El modelo de nutrición depende de un mayor número de variables con menor impacto individual, lo que incrementa la complejidad del problema.
Esto explica por qué el desempeño en entrenamiento es significativamente superior al de nutrición.

Interpretabilidad del modelo:

El uso de árboles de decisión permite una mayor interpretabilidad del modelo, ya que facilita identificar qué variables influyen en las decisiones. Esto es especialmente valioso en contextos reales, donde es importante justificar las recomendaciones generadas por el sistema.

ACTIVIDAD 8: Generación de predicciones

In [32]:
# Copia del test para no modificar el original
df_test_final = df_test.copy()

# Variables predictoras del train completo
X_full = df.drop(columns=["Plan_entrenamiento", "Plan_nutrición"])
y_full_ent = df["Plan_entrenamiento"]
y_full_nut = df["Plan_nutrición"]

# Asegurar consistencia de nombres de columnas
X_test_final = df_test_final.copy()

# Reentrenar mejor modelo para entrenamiento
best_tree_ent_final = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        criterion=grid_tree_ent.best_params_["model__criterion"],
        max_depth=grid_tree_ent.best_params_["model__max_depth"],
        min_samples_split=grid_tree_ent.best_params_["model__min_samples_split"],
        min_samples_leaf=grid_tree_ent.best_params_["model__min_samples_leaf"],
        random_state=42
    ))
])

best_tree_ent_final.fit(X_full, y_full_ent)

# Reentrenar mejor modelo para nutrición
best_tree_nut_final = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        criterion=grid_tree_nut.best_params_["model__criterion"],
        max_depth=grid_tree_nut.best_params_["model__max_depth"],
        min_samples_split=grid_tree_nut.best_params_["model__min_samples_split"],
        min_samples_leaf=grid_tree_nut.best_params_["model__min_samples_leaf"],
        random_state=42
    ))
])

best_tree_nut_final.fit(X_full, y_full_nut)

# Predicciones sobre test
pred_ent_test = best_tree_ent_final.predict(X_test_final)
pred_nut_test = best_tree_nut_final.predict(X_test_final)

# Agregar predicciones al archivo test
df_test_final["Plan_entrenamiento"] = pred_ent_test
df_test_final["Plan_nutrición"] = pred_nut_test

# Guardar CSV final
df_test_final.to_csv("Datos test Lab3 predicho.csv", sep=";", index=False, encoding="latin1")

print("Archivo generado correctamente: Datos test Lab3 predicho.csv")
print(df_test_final.head())

Archivo generado correctamente: Datos test Lab3 predicho.csv
   Edad     Genero   Peso  Altura    BMI           Objetivo Condicion_salud  \
0    32   Femenino   77.3    1.54  32.59  Ganancia muscular          Lesion   
1    29  Masculino   79.6    1.82  24.03            General         Ninguno   
2    33   Femenino  107.2    1.57  43.49            General    Hipertension   
3    42   Femenino  103.5    1.67  37.11            General            Asma   
4    29  Masculino   76.0    2.02  18.63  Ganancia muscular         Ninguno   

  Nivel_Actividad Nivel_experiencia Dieta_preferida  ...  Fumador  \
0            Bajo        Intermedio     Vegetariano  ...        0   
1            Alto          Avanzado     Vegetariano  ...        0   
2            Alto      Principiante          Vegano  ...        0   
3            Bajo      Principiante  No-Vegetariano  ...        0   
4            Bajo        Intermedio    Pescetariano  ...        0   

  Cigarrillos_dia  Alcohol  Alcohol_semana  Score

In [33]:
print(df_test_final.shape)
print(df_test_final[["Plan_entrenamiento", "Plan_nutrición"]].head(10))
print(df_test_final[["Plan_entrenamiento", "Plan_nutrición"]].isna().sum())

(302, 26)
  Plan_entrenamiento Plan_nutrición
0              Medio     Balanceado
1              Medio       Sin plan
2              Medio         Basico
3               Bajo       Sin plan
4               Bajo     Balanceado
5              Medio     Balanceado
6               Bajo     Balanceado
7              Medio     Balanceado
8            Ninguno       Sin plan
9            Ninguno       Sin plan
Plan_entrenamiento    0
Plan_nutrición        0
dtype: int64


Se generó el archivo final de predicciones utilizando el conjunto de datos de prueba, obteniendo un total de 302 registros con 26 variables. A cada instancia se le asignaron simultáneamente las predicciones correspondientes al plan de entrenamiento y al plan de nutrición, lo que evidencia el funcionamiento correcto del pipeline completo.

Al inspeccionar las primeras filas del resultado, se observa que las predicciones son coherentes con las características de entrada, mostrando combinaciones plausibles como planes “Medio” o “Bajo” para entrenamiento, y “Balanceado”, “Básico” o “Sin plan” para nutrición. Esto indica que el modelo está aplicando correctamente los patrones aprendidos durante el entrenamiento.

Adicionalmente, se verificó que no existen valores nulos en las columnas Plan_entrenamiento y Plan_nutrición, lo que confirma que el proceso de predicción fue exitoso para todos los registros del conjunto de prueba.

Finalmente, el archivo fue exportado correctamente en formato CSV bajo el nombre correspondiente, lo que permite su uso posterior para análisis, validación o integración en otros sistemas. Este resultado demuestra que el flujo completo del modelo, desde el preprocesamiento hasta la generación de predicciones, se ejecuta de manera consistente y sin errores.

9: BONO

In [34]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import pandas as pd
import numpy as np

X_multi = df.drop(columns=["Plan_entrenamiento", "Plan_nutrición"])
Y_multi = df[["Plan_entrenamiento", "Plan_nutrición"]]

X_train_m, X_val_m, Y_train_m, Y_val_m = train_test_split(
    X_multi,
    Y_multi,
    test_size=0.2,
    random_state=42
)

print(X_train_m.shape, X_val_m.shape)
print(Y_train_m.shape, Y_val_m.shape)

(7758, 24) (1940, 24)
(7758, 2) (1940, 2)


In [35]:
pipe_multi_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("model", MultiOutputClassifier(
        DecisionTreeClassifier(random_state=42)
    ))
])

In [36]:
param_grid_multi_tree = {
    "model__estimator__criterion": ["gini", "entropy"],
    "model__estimator__max_depth": [3, 5, 10, None],
    "model__estimator__min_samples_split": [2, 5, 10],
    "model__estimator__min_samples_leaf": [1, 2, 5]
}

In [37]:
def multioutput_f1_macro(estimator, X, y):
    y_pred = estimator.predict(X)
    
    f1_ent = f1_score(y.iloc[:, 0], y_pred[:, 0], average="macro")
    f1_nut = f1_score(y.iloc[:, 1], y_pred[:, 1], average="macro")
    
    return (f1_ent + f1_nut) / 2

In [38]:
grid_multi_tree = GridSearchCV(
    estimator=pipe_multi_tree,
    param_grid=param_grid_multi_tree,
    cv=5,
    scoring=multioutput_f1_macro,
    n_jobs=-1
)

grid_multi_tree.fit(X_train_m, Y_train_m)

print("Mejores parámetros (Multi-output árbol):")
print(grid_multi_tree.best_params_)

Mejores parámetros (Multi-output árbol):
{'model__estimator__criterion': 'entropy', 'model__estimator__max_depth': 10, 'model__estimator__min_samples_leaf': 1, 'model__estimator__min_samples_split': 2}


In [39]:
Y_pred_m = grid_multi_tree.predict(X_val_m)

pred_ent_m = Y_pred_m[:, 0]
pred_nut_m = Y_pred_m[:, 1]

print("Reporte multietiqueta - Plan Entrenamiento:")
print(classification_report(Y_val_m.iloc[:, 0], pred_ent_m, zero_division=0))

print("\nReporte multietiqueta - Plan Nutrición:")
print(classification_report(Y_val_m.iloc[:, 1], pred_nut_m, zero_division=0))

Reporte multietiqueta - Plan Entrenamiento:
              precision    recall  f1-score   support

        Alto       1.00      0.99      1.00       137
        Bajo       1.00      1.00      1.00       479
       Medio       1.00      1.00      1.00       599
     Ninguno       1.00      1.00      1.00       725

    accuracy                           1.00      1940
   macro avg       1.00      1.00      1.00      1940
weighted avg       1.00      1.00      1.00      1940


Reporte multietiqueta - Plan Nutrición:
               precision    recall  f1-score   support

   Balanceado       1.00      1.00      1.00       819
       Basico       0.99      0.99      0.99       465
Especializado       1.00      0.99      1.00       125
     Sin plan       1.00      1.00      1.00       531

     accuracy                           1.00      1940
    macro avg       1.00      0.99      1.00      1940
 weighted avg       1.00      1.00      1.00      1940



In [40]:
resultado_bono = pd.DataFrame([
    [
        "Multi-output árbol",
        "Entrenamiento",
        precision_score(Y_val_m.iloc[:, 0], pred_ent_m, average="macro"),
        recall_score(Y_val_m.iloc[:, 0], pred_ent_m, average="macro"),
        f1_score(Y_val_m.iloc[:, 0], pred_ent_m, average="macro")
    ],
    [
        "Multi-output árbol",
        "Nutrición",
        precision_score(Y_val_m.iloc[:, 1], pred_nut_m, average="macro"),
        recall_score(Y_val_m.iloc[:, 1], pred_nut_m, average="macro"),
        f1_score(Y_val_m.iloc[:, 1], pred_nut_m, average="macro")
    ]
], columns=["Modelo", "Objetivo", "Precision", "Recall", "F1-score"])

resultado_bono

,Modelo,Objetivo,Precision,Recall,F1-score
0,Multi-output árbol,Entrenamiento,0.999583,0.998175,0.998876
1,Multi-output árbol,Nutrición,0.996160,0.994297,0.995224


In [41]:
comparacion_final = pd.concat([results, resultado_bono], ignore_index=True)
comparacion_final

,Modelo,Objetivo,Precision,Recall,F1-score
0,Logística,Entrenamiento,0.892663,0.887893,0.889878
1,Árbol,Entrenamiento,1.000000,1.000000,1.000000
2,Logística,Nutrición,0.812930,0.785315,0.797529
3,Árbol,Nutrición,0.996160,0.994297,0.995224
4,Multi-output árbol,Entrenamiento,0.999583,0.998175,0.998876
5,Multi-output árbol,Nutrición,0.996160,0.994297,0.995224


Como extensión del laboratorio, se implementó un modelo de clasificación multietiqueta utilizando MultiOutputClassifier con un árbol de decisión como estimador base. Este enfoque permitió predecir simultáneamente las variables Plan_entrenamiento y Plan_nutrición dentro de un mismo pipeline, integrando preprocesamiento y entrenamiento en una única arquitectura.

Los resultados obtenidos muestran un desempeño extremadamente alto en ambas salidas. Para Plan_entrenamiento, el modelo multietiqueta alcanzó una precisión cercana a 0.9996, un recall de 0.9982 y un F1-score de 0.9989. Para Plan_nutrición, obtuvo una precisión de 0.9962, un recall de 0.9943 y un F1-score de 0.9952. Estas métricas indican que el modelo logra reproducir con gran exactitud las recomendaciones presentes en los datos.

Al comparar este enfoque con los modelos entrenados por separado, se observa que el desempeño es prácticamente equivalente al del mejor modelo individual. En Plan_entrenamiento, el árbol de decisión individual obtuvo un F1-score de 1.0000, mientras que el modelo multietiqueta alcanzó 0.9989. En Plan_nutrición, ambos enfoques lograron el mismo F1-score aproximado de 0.9952. Esto indica que el enfoque multietiqueta mantiene un nivel de desempeño muy similar al de los modelos separados.

La principal ventaja del modelo multietiqueta no radica en una mejora significativa de las métricas, sino en la posibilidad de manejar ambos objetivos dentro de una sola estructura de entrenamiento y predicción. Esto simplifica la implementación del sistema, facilita su mantenimiento y permite generar ambas recomendaciones de manera simultánea.

Sin embargo, es importante tener en cuenta que en scikit-learn MultiOutputClassifier no modela explícitamente dependencias complejas entre etiquetas, sino que entrena un clasificador por cada salida dentro de una misma envoltura. Por esta razón, su ventaja principal está en la organización conjunta del flujo de trabajo y no necesariamente en un aumento automático del rendimiento predictivo.

En conclusión, la actividad extra demuestra que es posible construir un modelo multietiqueta con un desempeño prácticamente igual al de los mejores modelos individuales. Esto lo convierte en una alternativa muy atractiva cuando se busca un sistema más unificado y sencillo de implementar, sin sacrificar calidad en las predicciones.